# Changing the simplex transform in NumPyro

NumPyro's autoguides (e.g. `AutoNormal`) pick the transform for each parameter with
`biject_to(distribution.support)`. For `dist.Dirichlet` this is `StickBreakingTransform`. To use a
different transform:

1. write it as a `Transform` subclass (`ParameterFreeTransform` if it has no parameters),
2. register it for a new simplex constraint with `biject_to.register`,
3. use a `Dirichlet` subclass whose `support` is that constraint.

As an example we use the transform Stan uses for `simplex`: a softmax of an isometric log-ratio (ILR) basis.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import numpyro.distributions as dist
from numpyro.distributions import constraints, transforms
from numpyro.distributions.transforms import biject_to

Default transform:

In [2]:
biject_to(dist.Dirichlet(jnp.ones(3)).support)

1. The new transform: theta = softmax(V y), with V an orthonormal zero-sum basis.

In [3]:
def ilr_basis(K):
    V = np.zeros((K, K - 1))
    for i in range(1, K):
        V[:i, i - 1] = 1 / np.sqrt(i * (i + 1))
        V[i, i - 1] = -i / np.sqrt(i * (i + 1))
    return jnp.asarray(V)


class ILRSimplexTransform(transforms.ParameterFreeTransform):
    domain = constraints.real_vector
    codomain = constraints.simplex

    def __call__(self, x):
        return jax.nn.softmax(x @ ilr_basis(x.shape[-1] + 1).T, axis=-1)

    def _inverse(self, y):
        return jnp.log(y) @ ilr_basis(y.shape[-1])

    def log_abs_det_jacobian(self, x, y, intermediates=None):
        return jnp.sum(jnp.log(y), axis=-1) + 0.5 * jnp.log(y.shape[-1])

    def forward_shape(self, shape):
        return shape[:-1] + (shape[-1] + 1,)

    def inverse_shape(self, shape):
        return shape[:-1] + (shape[-1] - 1,)

2–3. Register it for a new constraint, and give a Dirichlet that support.

In [4]:
class _ILRSimplex(type(constraints.simplex)):
    pass


@biject_to.register(_ILRSimplex)
def _(constraint):
    return ILRSimplexTransform()


class DirichletILR(dist.Dirichlet):
    support = _ILRSimplex()

Changed transform: use `DirichletILR` in the model wherever you would use `dist.Dirichlet`.

In [5]:
biject_to(DirichletILR(jnp.ones(3)).support)